In [ ]:
%matplotlib inline
import numpy as np
import torch
import sys
import os
import h5py
import matplotlib.pyplot as plt
import matplotlib.patches as patches
sys.path.append('../../')

from libs import runner, dataset, stats_eval, cond_gen, utils
from configs import *
from libs.lib_svd import Inpainting_custom
import libs.inst_eval as inst_eval



CONFIG_DICT = { 
                    'OneObs2D_ds1_10M': OneObs2D_ds1_10M.config_dict,
               }

In [ ]:
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

configname = "OneObs2D_ds1_10M"
osp_method= "qr"
threshold = None
start = 0
stop = 25000
step = 50
snaps = stop - start  

print(f"selected config ={configname}")
config_dict = CONFIG_DICT[configname]
config = runner.dict2namespace(config_dict)

In [ ]:
data, u_max, u_min, v_max, v_min, x, y ,t = cond_gen.get_data(config, datatype= "Train" ) #Test
u_max, u_min, v_max, v_min

In [ ]:
data_for_qr = data[start:stop:step]
u_mag = np.sqrt(data_for_qr[:, 0, :, :]**2 + data_for_qr[:, 1, :, :]**2)
u_mag.shape

In [ ]:
from scipy.linalg import qr

u_mag = np.expand_dims(u_mag , axis = 1)
print(u_mag.shape)

nt, nc, nx, ny = u_mag.shape

u_mag_qr = u_mag.reshape(nt, -1)

print(u_mag_qr.shape)

Q, R, P = qr(u_mag_qr, pivoting=True)

print(Q.shape, R.shape, P.shape)

In [ ]:
R_diag = np.abs(np.diag(R))

R_diag = R_diag.reshape(-1, 1)
plt.plot(range(len(R_diag)), R_diag)
plt.tick_params(axis='y', labelsize=12)  # Change font size for y-axis ticks
plt.tick_params(axis='x', labelsize=12)  # Change font size for x-axis ticks
plt.grid('on')

In [ ]:
def plot_inst_comparison_qr(num=0, P=P, num_sensors_to_mark=10, test_data=None, x_axis=None, y_axis=None, comp=None, colormap='viridis'):

        def validate_inputs():
            if test_data is None:
                raise ValueError("test_data must be provided.")
            if x_axis is None or y_axis is None:
                raise ValueError("x_axis and y_axis must be provided.")
            if comp not in ["streamwise", "wall-normal", "mag"]:
                raise ValueError("comp must be either 'streamwise' or 'wall-normal'.")

        def plot_subplot(ax, data, P, num_sensors_to_mark=num_sensors_to_mark, title=None):
            im = ax.imshow(data.T, cmap=colormap, extent=extent, origin='lower', aspect='auto', vmin=vmin, vmax=vmax)
            ax.set_title(title)
            ax.set_xlabel(r'$\frac{x}{h}$', fontsize=fontsize)
            ax.set_ylabel(r'$\frac{y}{h}$', fontsize=fontsize)
            obstacle = patches.Rectangle((pos_x, pos_y), width, height, linewidth=2, edgecolor='k', facecolor='k')
            ax.add_patch(obstacle)
            
            return im

        def mark_optimal_sensors_qr(ax, P, num_sensors_to_mark):
            """
            Marks the most important pixels on the provided axis according to the ranked indices.

            Parameters:
            ax (matplotlib.axes.Axes): The axis on which to plot the image and markers.
            P (np.ndarray): Ranked indices of shape (n_pixels,) indicating pixel importance.
            num_sensors_to_mark (int): The number of top pixels to mark.

            Returns:
            matplotlib.axes.Axes: The axis with the image and marked pixels.
            """
            # Get the first num_sensors_to_mark important pixel indices
            important_indices = P[:num_sensors_to_mark]
            
            # Convert flat indices to 2D coordinates (assuming image shape is 288x96)
            important_coords = [(index // test_data.shape[-1], index % test_data.shape[-1]) for index in important_indices]
            
            # Mark the important pixels on the image
            for x,y in important_coords:
                x_unit, y_unit, _ = utils.convert_pixel2unit(x_pixel=x, y_pixel=y, ds_ratio=config.dataset.ds_ratio)
                #print(f"{x} : {x_unit}, {y}: {y_unit}")
                ax.plot(x_unit, y_unit, 'ro',markersize=1)  # 'ro' means red color, circle marker

            return ax

        fontsize = 14
        # Validate inputs
        validate_inputs()

        # Obstacle dimensions & location
        pos_x, pos_y = -0.125, 0  # x position, y position
        width, height = 0.25, 1  # width, height of the obstacle

        fig, axs = plt.subplots(1, 3, figsize=(24, 4))

        # Define the extent based on x_axis and y_axis
        extent = [x_axis.min(), x_axis.max(), y_axis.min(), y_axis.max()]

        # Select the appropriate channel based on the component
        channel=0

        # Calculate the vmin and vmax based on test_data
        print(test_data.shape)
        vmin = np.min(test_data[num, channel, :, :])
        vmax = np.max(test_data[num, channel, :, :])

        # Plot the data
        im0 = plot_subplot(axs[0], test_data[num, channel, :, :], 'Test Data 1')
        im1 = plot_subplot(axs[1], test_data[num+1, channel, :, :], 'Test Data 2')
        im2 = plot_subplot(axs[2], test_data[num+2, channel, :, :], 'Test Data 3')

        axs[0] = mark_optimal_sensors_qr(axs[0], P, num_sensors_to_mark=num_sensors_to_mark)
        axs[1] = mark_optimal_sensors_qr(axs[1], P, num_sensors_to_mark=num_sensors_to_mark+1000)
        axs[2] = mark_optimal_sensors_qr(axs[2], P, num_sensors_to_mark=num_sensors_to_mark+1500)

        # Add a single colorbar for all subplots
        cbar = fig.colorbar(im0, ax=axs, orientation='vertical', fraction=0.02, pad=0.04)
        cbar.set_label(rf"${'u' if channel == 0 else 'v'}'$")

        # Adjust layout and display the figure
        #plt.tight_layout()
        plt.show()
        plt.close(fig)

In [ ]:
plot_inst_comparison_qr(num=0, P=P, num_sensors_to_mark=50, test_data=u_mag, x_axis=x, y_axis=y, comp="streamwise", colormap='viridis')

In [ ]:
#Select Sensors from the mask region
    #Get the co-ordinates of the qr sensors
    #Get the test data on which qr-pivoting must be tested
    #Get the mask for ex, 30%
    #Replace qr sensor, co-ordinates with 1
    #multiply the mask with this array



In [ ]:
def get_coords_optimal_sensors_qr(P=None, gtruth=None, num_sensors_to_mark=None):
    """
    Gets the qr sensors in pixel space according to the ranked indices.

    Parameters:
    P (np.ndarray): Ranked indices of shape (n_pixels,) indicating pixel importance.
    gtruth (np.ndarray): Gtruth data (DNS).

    Returns:
    matplotlib.axes.Axes: The axis with the image and marked pixels.
    """
    if num_sensors_to_mark:
        important_indices = P[:num_sensors_to_mark]
    else:
        important_indices = P

    # Convert flat indices to 2D coordinates (assuming as per gtruth shape)
    #for index in important_indices:
    #    print(f"{index // gtruth.shape[-1]}, {index % gtruth.shape[-1]}")

    qr_coords_pixels = np.unravel_index(important_indices, (gtruth[0,0].shape))
    #qr_coords_pixels = [(index // gtruth.shape[-1], index % gtruth.shape[-1]) for index in important_indices]
    qr_coords_pixels = np.array(qr_coords_pixels)
    print(f"Shape of qr-coords: {qr_coords_pixels.T.shape}")

    return qr_coords_pixels.T



In [ ]:
qr_coords_pixels = get_coords_optimal_sensors_qr(P=P, gtruth=data_for_qr, num_sensors_to_mark=None)

In [ ]:
from libs.cond_gen import create_mask, plot_mask

mask_type = "from_ground_and_wall"
mask = 30

mask_tensor, gtruth_torch, seg_tensor = create_mask(data=data_for_qr, mask=mask, ds_ratio=config.dataset.ds_ratio, mask_type=mask_type)

In [ ]:
qr_sensors = np.zeros_like(mask_tensor.cpu().numpy())

select_from_top= 500

x_qr_pixels =  qr_coords_pixels[:select_from_top, 0]
y_qr_pixels =  qr_coords_pixels[:select_from_top, 1]

qr_sensors[:, x_qr_pixels, y_qr_pixels ] = 1

qr_mask = qr_sensors * mask_tensor.cpu().numpy()

num_sensors = np.count_nonzero(qr_mask[0] == 1)
print(f"Number of sensors:{num_sensors}")

In [ ]:
plot_mask(mask_tensor=qr_mask, mask=mask, x_axis=x, y_axis=y, mask_type=f"qr_{num_sensors}"+mask_type, segmentation=False, out_show=True, out_pdf=False)


In [ ]:
np.savez(f"./qr_mask-start-{start}-stop-{stop}-step-{step}-num_sensors{num_sensors}-bs-{nt}.npz", qr_mask=qr_mask, all_qr_sensors=qr_sensors, num_sensors=num_sensors, mask=mask)

#Sensor Heatmap

In [ ]:
mask_250 = qr_mask

In [ ]:
mask_350 = qr_mask

In [ ]:
mask_500 = qr_mask


In [ ]:
mask_350.shape, mask_500.shape, mask_250.shape

In [ ]:
mask_importance = (mask_250 + mask_350 + mask_500)

In [ ]:
from libs.cond_gen import plot_mask, plot_sensor_importance

num_pixels = [0,22, 38, 71]

In [ ]:
plot_sensor_importance(sensor_importance_array=mask_importance, x_axis=x, y_axis=y, plot_name="QR", num_sensors = num_pixels, out_show=True, out_pdf=True)